# S&P 500 Returns and U.S. Macroeconomic Indicators — OLS Regression

**Objective:** Specify and estimate an OLS regression to test whether unemployment changes explain S&P 500 returns, and whether this relationship differs during NBER-defined recessions (via an interaction term). Diagnose residuals for heteroskedasticity and autocorrelation, and re-estimate with robust standard errors if needed.

## 1. Load Dataset and Recreate Stationary Variables
In this section, it is necessary to work with the variables `unemployment_rate`, `fed_rate`, `indpro` and `gs10` in their first-differenced form (confirmed stationary in notebook 03)

In [1]:
import pandas as pd

df = pd.read_csv("sp500_macro_monthly_1990_2026.csv", index_col=0, parse_dates=True)

df.head()

,unemployment_rate,fed_rate,indpro,gs10,usrec,sp500_close,inflation_yoy,sp500_log_return,log_indpro
1991-01-01,6.4,6.91,61.1355,8.09,1.0,343.929993,5.647059,4.067902,4.113093
1991-02-01,6.6,6.25,60.6838,7.85,1.0,367.070007,5.312500,6.511446,4.105677
1991-03-01,6.8,6.12,60.3346,8.11,1.0,375.220001,4.821151,2.195994,4.099906
1991-04-01,6.7,5.91,60.4938,8.04,0.0,375.339996,4.809930,0.031975,4.102541
1991-05-01,6.9,5.78,61.0633,8.07,0.0,389.829987,5.034857,3.787844,4.111911


In [2]:
vars_to_diff = ['unemployment_rate', 'fed_rate', 'indpro', 'gs10']

# Recreate first-differenced variables (confirmed stationary in notebook 03)
for col in vars_to_diff:
    df[f"{col}_diff"] = df[col].diff()

df = df.dropna(subset=[f"{col}_diff" for col in vars_to_diff])

print(df.shape)
print(df.columns.tolist())

(425, 13)
['unemployment_rate', 'fed_rate', 'indpro', 'gs10', 'usrec', 'sp500_close', 'inflation_yoy', 'sp500_log_return', 'log_indpro', 'unemployment_rate_diff', 'fed_rate_diff', 'indpro_diff', 'gs10_diff']


## 2. Multicollinearity Check (VIF)

Prior to specifying the OLS model, evaluating multicollinearity is essential given the anticipated correlation among `fed_rate_diff`, `gs10_diff`, and `inflation_yoy`. Fed rate reacts to inflation, and the 10-year yields embeds inflation expectations. 

In [3]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

X = sm.add_constant(df[['unemployment_rate_diff', 'fed_rate_diff', 'indpro_diff', 'gs10_diff', 'inflation_yoy']])

vif_data = pd.DataFrame()
vif_data["variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

                 variable       VIF
0                   const  4.149107
1  unemployment_rate_diff  2.257840
2           fed_rate_diff  1.122552
3             indpro_diff  2.294594
4               gs10_diff  1.037298
5           inflation_yoy  1.042878


## 3. OLS Specification — Layer 1

**Research question:** Does unemployment explain S&P 500 returns, and does that relationship change during NBER-defined recessions?

**Model:**
sp500_log_return ~ unemployment_rate_diff + fed_rate_diff + indpro_diff + gs10_diff + inflation_yoy + unemployment_rate_diff:usrec

The interaction term (`unemployment_rate_diff × usrec`) allows the effect of unemployment on returns to differ between recession and expansion periods — the central hypothesis of this project.

In [4]:
import statsmodels.api as sm

# Build the interaction term explicitly
df['unemployment_usrec_interaction'] = df['unemployment_rate_diff'] * df['usrec']

X = df[['unemployment_rate_diff', 'fed_rate_diff', 'indpro_diff', 'gs10_diff', 
        'inflation_yoy', 'unemployment_usrec_interaction']]
X = sm.add_constant(X)

y = df['sp500_log_return']

model = sm.OLS(y, X, missing='drop').fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       sp500_log_return   R-squared:                       0.032
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     2.268
Date:                Sat, 15 Aug 2026   Prob (F-statistic):             0.0364
Time:                        01:06:16   Log-Likelihood:                -1211.6
No. Observations:                 425   AIC:                             2437.
Df Residuals:                     418   BIC:                             2466.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       

## 4. Residual Diagnostics

Testing heteroskedasticity (Breusch-Pagan) and autocorrelation (Breusch-Godfrey, confirming the preliminary Durbin-Watson result of 2.053) on the OLS residuals.

In [5]:
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey

# Heteroskedasticity
bp_stat, bp_p, _, _ = het_breuschpagan(model.resid, model.model.exog)
print(f"Breusch-Pagan: stat={bp_stat:.4f}, p-value={bp_p:.4f}")
print("Heteroskedasticity present" if bp_p < 0.05 else "Homoskedastic (OK)")

print()

# Autocorrelation (confirms/contradicts Durbin-Watson ≈ 2.053)
bg_stat, bg_p, _, _ = acorr_breusch_godfrey(model, nlags=4)
print(f"Breusch-Godfrey: stat={bg_stat:.4f}, p-value={bg_p:.4f}")
print("Autocorrelation present" if bg_p < 0.05 else "No autocorrelation (OK)")

Breusch-Pagan: stat=17.2191, p-value=0.0085
Heteroskedasticity present

Breusch-Godfrey: stat=4.4717, p-value=0.3459
No autocorrelation (OK)


## 5. Re-estimate with Robust Standard Errors

Given confirmed heteroskedasticity (Breusch-Pagan p=0.0085), the model is re-estimated using HC3 robust standard errors. Coefficients remain identical; only standard errors, t-statistics, and p-values are corrected.

In [ ]:
model_robust = sm.OLS(y,X,missing='drop').fit(cov_type='HC3')

print(model_robust.summary())

                            OLS Regression Results                            
Dep. Variable:       sp500_log_return   R-squared:                       0.032
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                    0.9283
Date:                Sat, 15 Aug 2026   Prob (F-statistic):              0.474
Time:                        01:06:16   Log-Likelihood:                -1211.6
No. Observations:                 425   AIC:                             2437.
Df Residuals:                     418   BIC:                             2466.
Df Model:                           6                                         
Covariance Type:                  HC3                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       